<a href="https://colab.research.google.com/github/redzabdullah/EBSCOHOst_EconLit/blob/main/EBSCOHost_EconLit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

# ==========================================================
# EconLit XML → JEL Classification Mapping
# Source: American Economic Association
# Method: Exact descriptor matching (no AI or fuzzy matching)
# ==========================================================

import re
import io
import hashlib
import requests
import pandas as pd
import xml.etree.ElementTree as ET

from bs4 import BeautifulSoup
from collections import defaultdict
from datetime import datetime, timezone
from google.colab import files

AEA_URL = (
    "https://www.aeaweb.org/econlit/"
    "jelCodes.php?view=econlit"
)

def normalize(text):
    """Normalize whitespace/case without changing punctuation."""
    return " ".join((text or "").casefold().split())

# ----------------------------------------------------------
# STEP A: Upload and parse EBSCO XML
# ----------------------------------------------------------

uploaded = files.upload()
xml_filename = next(
    (name for name in uploaded if name.lower().endswith(".xml")),
    None
)

if not xml_filename:
    raise ValueError("Please upload an EBSCO XML file.")

xml_bytes = uploaded[xml_filename]
xml_hash = hashlib.sha256(xml_bytes).hexdigest()

root = ET.fromstring(xml_bytes)
records = [root] if root.tag == "record" else root.findall(".//record")

articles = []
assignments = []

for index, record in enumerate(records, start=1):
    def field(tag):
        return (record.findtext(tag) or "").strip()

    accession = field("accessionNumber")
    subjects = field("subjects")

    articles.append({
        "record_order": index,
        "accessionNumber": accession,
        "title": field("title"),
        "authors": field("authors"),
        "publicationDate": field("publicationDate"),
        "source": field("source"),
        "doi": field("doi"),
        "original_subjects": subjects
    })

    # A separator is a semicolon WITH surrounding spaces.
    # Internal semicolons in descriptors must be preserved.
    descriptors = [
        s.strip()
        for s in re.split(r"\s+;\s+", subjects)
        if s.strip()
    ]

    for position, descriptor in enumerate(descriptors, start=1):
        assignments.append({
            "accessionNumber": accession,
            "descriptor_position": position,
            "descriptor": descriptor,
            "normalized": normalize(descriptor)
        })

articles_df = pd.DataFrame(articles)
assign_df = pd.DataFrame(assignments)

print("XML records:", len(articles_df))
print("Subject assignments:", len(assign_df))
print("Unique descriptors:", assign_df["normalized"].nunique())
print("Unique accession numbers:",
      articles_df["accessionNumber"].nunique())

# Basic validation
assert articles_df["accessionNumber"].ne("").all()
assert articles_df["accessionNumber"].is_unique

# ----------------------------------------------------------
# STEP B: Retrieve official AEA vocabulary
# ----------------------------------------------------------

response = requests.get(
    AEA_URL,
    headers={"User-Agent": "Mozilla/5.0"},
    timeout=40
)
response.raise_for_status()

retrieved_at = datetime.now(timezone.utc).isoformat()
html = response.text

soup = BeautifulSoup(html, "html.parser")
code_pattern = re.compile(r"^[A-Z][0-9]{2}$")

aea_rows = []

for row in soup.select("tr"):
    cells = [
        cell.get_text(" ", strip=True)
        for cell in row.find_all(["td", "th"], recursive=False)
    ]

    # Locate the classification code and its description.
    for i, value in enumerate(cells[:-1]):
        cleaned_code = re.sub(r"[†‡*]", "", value).strip()

        if code_pattern.fullmatch(cleaned_code):
            description = cells[i + 1].strip()
            description = re.sub(r"\s*[†‡]\s*$", "", description)

            if description:
                aea_rows.append({
                    "jel_code": cleaned_code,
                    "aea_descriptor": description,
                    "normalized": normalize(description)
                })
            break

aea_df = pd.DataFrame(
    aea_rows,
    columns=["jel_code", "aea_descriptor", "normalized"]
).drop_duplicates()

# Stop rather than silently use an incomplete vocabulary.
if len(aea_df) < 500:
    raise RuntimeError(
        f"Only {len(aea_df)} AEA entries parsed. "
        "The website layout may have changed or access "
        "may have been restricted. Inspect the HTML "
        "before proceeding."
    )

print("\nAEA entries extracted:", len(aea_df))
print("AEA retrieval time (UTC):", retrieved_at)

# ----------------------------------------------------------
# STEP C: Build exact-match lookup
# ----------------------------------------------------------

lookup = defaultdict(set)

for row in aea_df.itertuples():
    lookup[row.normalized].add(row.jel_code)

def classify(descriptor):
    codes = sorted(lookup.get(normalize(descriptor), set()))

    if len(codes) == 1:
        return "exact_unique", codes[0]

    if len(codes) > 1:
        return "ambiguous", "; ".join(codes)

    return "unmatched", ""

assign_df[["match_status", "jel_code"]] = assign_df[
    "descriptor"
].apply(lambda s: pd.Series(classify(s)))

# ----------------------------------------------------------
# STEP D: Construct descriptor audit table
# ----------------------------------------------------------

audit_df = (
    assign_df.groupby(
        ["descriptor", "normalized", "match_status", "jel_code"],
        as_index=False
    )
    .agg(
        assignment_count=("accessionNumber", "size"),
        article_count=("accessionNumber", "nunique")
    )
    .sort_values(
        ["assignment_count", "descriptor"],
        ascending=[False, True]
    )
)

audit_df["aea_source"] = AEA_URL
audit_df["aea_retrieved_utc"] = retrieved_at

# ----------------------------------------------------------
# STEP E: Enrich article-level metadata
# ----------------------------------------------------------

enriched = []

for article in articles:
    accession = article["accessionNumber"]
    subset = assign_df[
        assign_df["accessionNumber"] == accession
    ].sort_values("descriptor_position")

    matched = subset[subset["match_status"] == "exact_unique"]
    unresolved = subset[subset["match_status"] != "exact_unique"]

    codes = list(dict.fromkeys(matched["jel_code"].tolist()))

    enriched.append({
        **article,
        "jel_codes_mapped": "; ".join(codes),
        "descriptor_count": len(subset),
        "mapped_count": len(matched),
        "unresolved_count": len(unresolved),
        "all_descriptors_mapped": len(unresolved) == 0
    })

enriched_df = pd.DataFrame(enriched)

# ----------------------------------------------------------
# STEP F: Calculate observed coverage
# ----------------------------------------------------------

status_counts = assign_df["match_status"].value_counts()

exact = int(status_counts.get("exact_unique", 0))
ambiguous = int(status_counts.get("ambiguous", 0))
unmatched = int(status_counts.get("unmatched", 0))
total = len(assign_df)

coverage = 100 * exact / total if total else 0

print("\n========== MAPPING RESULTS ==========")
print("Records:", len(enriched_df))
print("Subject assignments:", total)
print("Unique descriptors:", len(audit_df))
print("Exact assignments:", exact)
print("Ambiguous assignments:", ambiguous)
print("Unmatched assignments:", unmatched)
print(f"Exact-match coverage: {coverage:.2f}%")
print(
    "Fully mapped records:",
    int(enriched_df["all_descriptors_mapped"].sum())
)

print("\nFirst five enriched records:")
display(enriched_df[
    ["accessionNumber", "title", "jel_codes_mapped",
     "all_descriptors_mapped"]
].head())

print("\nUnresolved descriptors:")
display(audit_df[
    audit_df["match_status"] != "exact_unique"
])

# ----------------------------------------------------------
# STEP G: Export reproducible results
# ----------------------------------------------------------

enriched_df.to_csv(
    "econlit_greenwashing_enriched.csv",
    index=False, encoding="utf-8-sig"
)

audit_df.to_csv(
    "econlit_descriptor_mapping_audit.csv",
    index=False, encoding="utf-8-sig"
)

report_df = pd.DataFrame([{
    "xml_filename": xml_filename,
    "xml_sha256": xml_hash,
    "aea_url": AEA_URL,
    "aea_retrieved_utc": retrieved_at,
    "aea_entries": len(aea_df),
    "records": len(enriched_df),
    "unique_descriptors": len(audit_df),
    "subject_assignments": total,
    "exact_assignments": exact,
    "ambiguous_assignments": ambiguous,
    "unmatched_assignments": unmatched,
    "exact_match_rate_pct": round(coverage, 2),
    "fully_mapped_records": int(
        enriched_df["all_descriptors_mapped"].sum()
    )
}])

report_df.to_csv(
    "econlit_mapping_report.csv",
    index=False, encoding="utf-8-sig"
)

print("\nThree CSV output files created.")

files.download("econlit_greenwashing_enriched.csv")
files.download("econlit_descriptor_mapping_audit.csv")
files.download("econlit_mapping_report.csv")


Saving EBSCO-Metadata-11_11_2026_greenwashing.xml to EBSCO-Metadata-11_11_2026_greenwashing.xml
XML records: 50
Subject assignments: 283
Unique descriptors: 73
Unique accession numbers: 50

AEA entries extracted: 859
AEA retrieval time (UTC): 2026-11-11T03:05:08.497939+00:00

========== MAPPING RESULTS ==========
Records: 50
Subject assignments: 283
Unique descriptors: 73
Exact assignments: 283
Ambiguous assignments: 0
Unmatched assignments: 0
Exact-match coverage: 100.00%
Fully mapped records: 50

First five enriched records:


,accessionNumber,title,jel_codes_mapped,all_descriptors_mapped
0,2249157,Will Informal Environmental Regulation Become ...,M14; Q53; Q58,True
1,2248793,Evaluating Sustainability in Local Public Proj...,H75; H76; R53; R58,True
2,2249072,ESG Greenwashing and Stock Price Crash Risk: A...,G12; M14; Q50,True
3,2249377,Does Political Connection Stifle Corporate Hyp...,D72; M14; M31; O14; O17; P23; P26,True
4,2241508,Greenwashing as a Corporate Strategy: A Biblio...,D82; G12; G32; G34; L21; M14; Q52,True



Unresolved descriptors:


,descriptor,normalized,match_status,jel_code,assignment_count,article_count,aea_source,aea_retrieved_utc



Three CSV output files created.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>